# Onderzoeksvragen


## data laden en preprocessen





In [1]:
import pandas as pd
import networkx as nx
import re
from collections import Counter
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt


In [2]:
%%time
df= pd.read_csv('LidoLinks.csv.gz', header=None)
print(df.shape)
df.head()


(7355861, 2)
CPU times: total: 11 s
Wall time: 11 s


,0,1
0,jurisprudentie/id/ECLI:NL:RBAMS:2015:6776,bwb/id/BWBR0001854/2967974/1984-05-01/1984-05-01
1,jurisprudentie/id/ECLI:NL:RBOBR:2018:4470,bwb/id/BWBR0005290/2913524/2018-07-28/2018-07-28
2,jurisprudentie/id/ECLI:NL:RBAMS:2019:4496,bwb/id/BWBR0001854/2963144/1986-05-19/1986-05-19
3,jurisprudentie/id/ECLI:NL:RBOVE:2024:2366,bwb/id/BWBR0001827/3178144/2002-01-01/2002-01-01
4,jurisprudentie/id/ECLI:NL:RBUTR:2006:BA7009,bwb/id/BWBR0002375/2060994/2006-01-01/2006-01-01


In [3]:
%%time
# haal de datums van de id's af. Dit werkt alleen op de bwb/id knopen
df[1]=df[1].str.replace(r'(bwb.*?)(/[-0-9]+)?/[-0-9]+$',r'\1',regex=True) 
df[0]=df[0].str.replace(r'(bwb.*?)(/[-0-9]+)?/[-0-9]+$',r'\1',regex=True) 

CPU times: total: 25.4 s
Wall time: 25.9 s


In [4]:
df['groep_s'] = df[0].str.replace(r'/id.*', r'', regex=True)
df['groep_t'] = df[1].str.replace(r'/id.*', r'', regex=True)

## 11. Onderzoeksvraag

Hoe is de verdeling van invloed in het citatienetwerk van Rechtspraak.nl, gemeten via in-degree, PageRank en Marc in-degree, en in hoeverre is deze invloed geconcentreerd in een klein aantal uitspraken?


1. Wat is de verdeling van het aantal ontvangen citaties (in-degree), PageRank en Marc in-degree over uitspraken in het citatienetwerk?
2. In hoeverre is invloed ongelijk verdeeld over uitspraken, en verschilt deze ongelijkheid tussen de drie maten?
3. In hoeverre vertonen deze distributies kenmerken van een schaalvrije (power-law) verdeling?
4. In hoeverre stemmen de drie maten overeen in welke uitspraken zij als invloedrijk identificeren, en wat betekent dit voor de robuustheid van de gevonden concentratiepatronen?
5. Wat betekenen de gevonden patronen voor de concentratie van invloed binnen het citatienetwerk?


In [8]:
# G_jur: citatienetwerk (jurisprudentie -> jurisprudentie)
mask_jur = (df['groep_s'] == 'jurisprudentie') & (df['groep_t'] == 'jurisprudentie')
df_jur = df.loc[mask_jur, [0, 1]].copy()

# selfloops weghalen
self_loops = (df_jur[0] == df_jur[1]).sum()
df_jur = df_jur[df_jur[0] != df_jur[1]]

# dubbele edghes weghalen
n_before_dedup = len(df_jur)
df_jur = df_jur.drop_duplicates()
n_after_dedup = len(df_jur)

G_jur = nx.from_pandas_edgelist(df_jur, source=0, target=1, create_using=nx.DiGraph())

print('Citatienetwerk jur->jur:')
print(f'nodes:{G_jur.number_of_nodes():,}')
print(f'edges{G_jur.number_of_edges():,}')
print(f'zelfreferenties verwijderd:{self_loops:,}')
print(f'dubbele edges verwijderd:{n_before_dedup - n_after_dedup:,}')
print(f'dichtheid:{nx.density(G_jur):.2e}')


Citatienetwerk jur->jur:
nodes:674,589
edges1,353,664
zelfreferenties verwijderd:9
dubbele edges verwijderd:0
dichtheid:2.97e-06
